In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [2]:
from langchain.chat_models import init_chat_model
import os

model = init_chat_model("openai:qwen3.7-plus", temperature=0, base_url=os.getenv("ROOT_URL"))
print(type(model))

<class 'langchain_openai.chat_models.base.ChatOpenAI'>


In [3]:
from langchain_tavily import TavilySearch

web_search = TavilySearch(
    max_results=5,
    topic="general"
    )

In [6]:
from langgraph.checkpoint.sqlite import SqliteSaver

import sqlite3

connection = sqlite3.connect("resources/personal_chief.db", check_same_thread=False)
checkpointer = SqliteSaver(connection)
checkpointer.setup()


In [8]:
from langchain.agents import create_agent

system_prompt = """
你是一名私人厨师。收到用户提供的食材照片或清单后，请按以下流程操作：
1.识别和评估食材：若用户提供照片，首先辨识所有可见食材。基于食材的外观状态，评估其新鲜度与可用量，整理出一份“当前可用食材清单”。
2.智能食谱检索：优先调用 web_search 工具，以“可用食材清单”为核心关键词，查找可行菜谱。
3.多维度评估与排序：从营养价值和制作难度两个维度对检索到的候选食谱进行量化打分，并根据得分排序，制作简单且营养丰富的排名靠前。
4.结构化方案输出：把排序后的食谱整理为一份结构清晰的建议报告，要包含食谱信息、得分、推荐理由、食谱的参考图片，帮助用户快速做出决策。

请严格按照流程，优先调用 web_search 工具搜索食谱，搜索不到的情况下才能自己发挥。
"""

agent = create_agent(
    system_prompt=system_prompt,
    tools=[web_search],
    model=model,
    checkpointer=checkpointer)

In [9]:
from langchain.messages import HumanMessage

# 准备多模态消息，图片是网络上的冰箱食物图片
multimodal_message = HumanMessage(
    content=[
        {"type": "image",
         "url": "https://img.freepik.com/free-photo/arrangement-different-foods-organized-fridge_23-2149099882.jpg"},
        {"type": "text", "text": "帮我看看这些食材能做些什么？"}
    ])

config = {"configurable": {"thread_id": "6"}}

response = agent.invoke({"messages": [multimodal_message]}, config)

In [10]:
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'image', 'url': 'https://img.freepik.com/free-photo/arrangement-different-foods-organized-fridge_23-2149099882.jpg'}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}]
================================== Ai Message ==================================

## 🧊 食材识别结果

我已仔细识别了冰箱中的所有食材，整理如下：

| 类别 | 食材 | 新鲜度评估 |
|------|------|-----------|
| 🐟 蛋白质 | 三文鱼、鸡胸肉 | 色泽鲜亮，状态良好 |
| 🥦 蔬菜 | 西兰花、蘑菇、小番茄 | 饱满新鲜 |
| 🫑 彩椒 | 红甜椒、黄甜椒 | 色泽鲜艳 |
| 🥬 叶菜 | 生菜 | 翠绿新鲜 |
|  根茎 | 洋葱、白萝卜 | 状态良好 |

---

现在让我为您搜索最佳食谱方案！
Tool Calls:
  tavily_search (call_3705df3c4ca54328856db409)
 Call ID: call_3705df3c4ca54328856db409
  Args:
    query: 三文鱼 西兰花 蘑菇 健康食谱 简单做法
    search_depth: advanced
  tavily_search (call_99f3bdb90003456da6ed1006)
 Call ID: call_99f3bdb90003456da6ed1006
  Args:
    query: 鸡胸肉 彩椒 洋葱 低脂健康食谱 做法
    search_depth: advanced
  tavily_search (call_13b4b896da674c9d9404e2b5)
 Call ID: call_13b4b896da674c9d9404e2b5
  Args:
    query

In [11]:
response = agent.invoke({"messages": [HumanMessage(content="我喜欢第三道菜,可否详细介绍一下？")]}, config)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'image', 'url': 'https://img.freepik.com/free-photo/arrangement-different-foods-organized-fridge_23-2149099882.jpg'}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}]
================================== Ai Message ==================================

## 🧊 食材识别结果

我已仔细识别了冰箱中的所有食材，整理如下：

| 类别 | 食材 | 新鲜度评估 |
|------|------|-----------|
| 🐟 蛋白质 | 三文鱼、鸡胸肉 | 色泽鲜亮，状态良好 |
| 🥦 蔬菜 | 西兰花、蘑菇、小番茄 | 饱满新鲜 |
| 🫑 彩椒 | 红甜椒、黄甜椒 | 色泽鲜艳 |
| 🥬 叶菜 | 生菜 | 翠绿新鲜 |
|  根茎 | 洋葱、白萝卜 | 状态良好 |

---

现在让我为您搜索最佳食谱方案！
Tool Calls:
  tavily_search (call_3705df3c4ca54328856db409)
 Call ID: call_3705df3c4ca54328856db409
  Args:
    query: 三文鱼 西兰花 蘑菇 健康食谱 简单做法
    search_depth: advanced
  tavily_search (call_99f3bdb90003456da6ed1006)
 Call ID: call_99f3bdb90003456da6ed1006
  Args:
    query: 鸡胸肉 彩椒 洋葱 低脂健康食谱 做法
    search_depth: advanced
  tavily_search (call_13b4b896da674c9d9404e2b5)
 Call ID: call_13b4b896da674c9d9404e2b5
  Args:
    query